# 02 — SLAM Exploration

Use the robot's `explore()` method to autonomously map an unknown environment.

**What you'll learn**:
- Launch autonomous frontier-based exploration
- Monitor map building progress
- Access the occupancy grid map

**Prerequisites**: `pip install threewe[sim]`

In [ ]:
import asyncio
import numpy as np
from threewe import Robot

In [ ]:
robot = Robot(backend="gazebo", scene="office_v2")
robot.connect()

## Check Initial Map State

Before exploration, the map is mostly unknown (`-1` values).

In [ ]:
grid = robot.get_map()
total_cells = grid.data.size
known_cells = np.sum(grid.data >= 0)
print(f"Map size: {grid.data.shape}")
print(f"Resolution: {grid.resolution} m/cell")
print(f"Known cells: {known_cells}/{total_cells} ({100*known_cells/total_cells:.1f}%)")

## Launch Autonomous Exploration

The `explore()` method uses frontier-based exploration:
1. Find frontiers (boundaries between known and unknown space)
2. Navigate to the nearest frontier
3. Repeat until map coverage is sufficient or timeout

This wraps Nav2 + SLAM under the hood — you don't need to know ROS2.

In [ ]:
print("Starting autonomous exploration (60s timeout)...")
result = await robot.explore(timeout=60.0)

print(f"\nExploration complete!")
print(f"  Success: {result.success}")
print(f"  Coverage: {result.coverage:.1%}")
print(f"  Duration: {result.duration:.1f}s")
print(f"  Distance traveled: {result.distance:.2f}m")

## Inspect the Final Map

In [ ]:
final_grid = robot.get_map()
known_cells = np.sum(final_grid.data >= 0)
free_cells = np.sum(final_grid.data == 0)
occupied_cells = np.sum(final_grid.data == 100)

print(f"Final map statistics:")
print(f"  Total cells: {final_grid.data.size}")
print(f"  Known: {known_cells} ({100*known_cells/final_grid.data.size:.1f}%)")
print(f"  Free: {free_cells}")
print(f"  Occupied (walls/obstacles): {occupied_cells}")
print(f"  Map origin: ({final_grid.origin.x:.1f}, {final_grid.origin.y:.1f})")

## Visualize the Map (Optional)

The occupancy grid is a plain numpy array — use matplotlib or any plotting library.

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    display_map = np.where(final_grid.data == -1, 128, final_grid.data)
    ax.imshow(display_map, cmap="gray_r", origin="lower")
    ax.set_title("Occupancy Grid (white=free, black=wall, gray=unknown)")
    ax.set_xlabel(f"cells (resolution={final_grid.resolution}m)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for visualization: pip install matplotlib")

In [ ]:
robot.disconnect()